# M11.3 · Politika yinelemesi ve değiştirilmiş politika yinelemesi

Değeri değil politikayı doğrudan iyileştirmek: değerlendir → açgözlü iyileştir döngüsü, her adımda değerin monoton artması, sonlu sayıda adımda yakınsama ve ikisinin arasındaki değiştirilmiş politika yinelemesi (k değerlendirme taraması). Elle iki durumlu bir PI; saf Python'da Sutton'ın 4 × 4 ızgara dünyası ve rastgele politikanın ünlü değerleri; NumPy'da VI, PI ve MPI'nin iterasyon / yedekleme (backup) muhasebesi; PyTorch'ta Jack'in araba kiralama problemini Poisson geçişleriyle tam çözüp kitaptaki politika haritasını yeniden üretiyorsun; en sonda büyük MDP'lerde politika değerlendirmesi için doğrudan çözüm ile yinelemeli çözümü karşılaştırıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m11-3-policy-iteration.html](https://m-gonul.github.io/pytorch-lab/m11-3-policy-iteration.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m11_3_policy_iteration.py
CHECKS = {
 "b": "import numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def st():\n        ensure(need(ns, \"grid_step\")(1, 0) == (1, -1.0) and ns[\"grid_step\"](1, 1) == (5, -1.0) and ns[\"grid_step\"](4, 2) == (4, -1.0), \"duvarlar\")\n    R_.test(\"grid_step\", st)\n\n    def ev():\n        V, sw = need(ns, \"evaluate\")([[0.25] * 4 for _ in range(16)])\n        want = [0, -14, -20, -22, -14, -18, -20, -20, -20, -20, -18, -14, -22, -20, -14, 0]\n        close(np.array(V), np.array(want, float), atol=1e-6, what=\"Sutton Şekil 4.1 (k = ∞)\")\n        return f\"{sw} yerinde tarama\"\n    R_.test(\"rastgele politikanın değeri\", ev)\n\n    def pi():\n        V, p, n = need(ns, \"policy_iteration\")()\n        dist = [min(r + c, 6 - r - c) for r, c in (divmod(s, 4) for s in range(16))]\n        close(np.array(V), -np.array(dist, float), atol=1e-6, what=\"en iyi değer = −(en yakın köşeye uzaklık)\")\n        ensure(n <= 4, f\"{n} iyileştirme\")\n        arrows = \"↑↓←→\"\n        grid = [\"\".join(\"■\" if s in (0, 15) else arrows[p[s].index(1.0)] for s in range(r * 4, r * 4 + 4)) for r in range(4)]\n        return f\"{n} iyileştirme; politika: \" + \" / \".join(grid)\n    R_.test(\"policy_iteration\", pi)\n",
 "c": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    rng = np.random.default_rng(0)\n    S, A = 60, 4\n    P = rng.random((A, S, S)) ** 6; P /= P.sum(-1, keepdims=True)\n    R = rng.normal(size=(S, A))\n\n    def same():\n        g = 0.95\n        V1, p1, *_ = need(ns, \"vi\")(P, R, g, 1e-10)\n        V2, p2, *_ = need(ns, \"pi_exact\")(P, R, g)\n        V3, p3, *_ = need(ns, \"mpi\")(P, R, g, 5, 1e-10)\n        close(V1, V2, atol=1e-7, what=\"VI ve PI\"); close(V3, V2, atol=1e-7, what=\"MPI ve PI\")\n        ensure(np.array_equal(p1, p2) and np.array_equal(p3, p2), \"aynı politika\")\n    R_.test(\"üçü aynı V* ve π*\", same)\n\n    def acct():\n        rows = []\n        for g in (0.9, 0.99):\n            _, _, i1, b1 = ns[\"vi\"](P, R, g, 1e-8)\n            _, _, i2, b2 = ns[\"pi_exact\"](P, R, g)\n            row = [g, f\"{i1} / {b1:,}\", f\"{i2} / {b2:,}\"]\n            for k in (2, 5, 20):\n                _, _, i3, b3 = ns[\"mpi\"](P, R, g, k, 1e-8)\n                row.append(f\"{i3} / {b3:,}\")\n            rows.append(row)\n        ensure(int(rows[1][2].split(\" / \")[0]) < int(rows[1][1].split(\" / \")[0]), \"PI daha az iterasyon\")\n        R_.table(\"İterasyon / backup (60 durum, 4 eylem, ε = 1e-8)\", [\"γ\", \"VI\", \"PI (tam çözüm)\", \"MPI k = 2\", \"MPI k = 5\", \"MPI k = 20\"], rows,\n                 note=\"γ → 1'de VI'nın iterasyonu ~1/(1−γ) büyür; PI birkaç iterasyonda biter ama her biri bir doğrusal sistem.\")\n    R_.test(\"hesap muhasebesi\", acct)\n",
 "d": "import numpy as np\nimport torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def loc():\n        T, r = need(ns, \"location\")(3, 3)\n        close(T.sum(1), np.ones(21), atol=1e-12, what=\"satırlar olasılık\")\n        ensure(r[0] == 0 and 29 < r[20] <= 30, f\"r(0) = 0, r(20) ≈ 30 bekleniyordu: {r[20]:.2f}\")\n        T0, _ = ns[\"location\"](3, 3)\n        close(T0[0, :4], np.array([np.exp(-3) * 3 ** k / np.prod(range(1, k + 1)) for k in range(4)]), atol=1e-12, what=\"araba yokken ertesi gün = iadeler\")\n    R_.test(\"location\", loc)\n\n    def jp():\n        V, pi, it = need(ns, \"jack_pi\")()\n        V, pi = np.asarray(V), np.asarray(pi)\n        ensure(pi[20, 0] == 5 and pi[0, 20] < 0 and pi[0, 0] == 0, \"kitaptaki π*'ın köşeleri: (20, 0) → 5, (0, 20) → negatif, (0, 0) → 0\")\n        ensure(it <= 6, f\"{it} iyileştirme\")\n        ensure(400 < V[0, 0] < 440 and V[20, 20] > V[0, 0], f\"V aralığı: {V.min():.0f}–{V.max():.0f}\")\n        rows = [\"\".join(f\"{v:+d}\" if v else \" 0\" for v in pi[n1]) for n1 in (20, 15, 10, 5, 0)]\n        R_.table(\"π* (satır: 1. şubedeki araba; sütun: 2. şube 0…20)\", [\"n₁\", \"taşınan araba (+: 1 → 2)\"], [[n, r] for n, r in zip((20, 15, 10, 5, 0), rows)],\n                 note=f\"{it} politika iyileştirmesi; V ∈ [{V.min():.0f}, {V.max():.0f}]. Sutton Şekil 4.2'deki bantlı yapı: 2. şubenin talebi yüksek, iadesi düşük olduğu için arabalar oraya taşınır.\")\n    R_.test(\"Jack'in araba kiralama problemi\", jp)\n",
 "e": "import torch\nfrom labcheck import close, need, ensure, bench, fmt_time\n\n\ndef check(ns, R_):\n    def ev():\n        idx, prob, R = need(ns, \"random_sparse_mdp\")(300, 3, 6, 1)\n        pi = torch.zeros(300, dtype=torch.long)\n        ar = torch.arange(300)\n        dense = torch.zeros(300, 300, dtype=torch.float64)\n        dense.index_put_((ar[:, None].expand(-1, 6), idx[0]), prob[0], accumulate=True)\n        r = R[:, 0]\n        a = need(ns, \"eval_direct\")(dense, r, 0.95)\n        b, n = need(ns, \"eval_iterative\")(idx[0], prob[0], r, 0.95)\n        close(b.numpy(), a.numpy(), atol=1e-7, what=\"yinelemeli = doğrudan\")\n        return f\"{n} tarama\"\n    R_.test(\"eval_direct ve eval_iterative\", ev)\n\n    def mp():\n        idx, prob, R = ns[\"random_sparse_mdp\"](200, 4, 5, 2)\n        V5, p5, _ = need(ns, \"mpi_sparse\")(idx, prob, R, 0.95, 5, 1e-10)\n        V1, p1, _ = ns[\"mpi_sparse\"](idx, prob, R, 0.95, 1, 1e-10)\n        close(V5.numpy(), V1.numpy(), atol=1e-7, what=\"MPI k = 5 ile VI aynı V*\")\n    R_.test(\"mpi_sparse\", mp)\n    if \"eval_direct\" not in ns:\n        return\n    rows = []\n    for S in (500, 1500, 3000):\n        idx, prob, R = ns[\"random_sparse_mdp\"](S, 1, 8, 3)\n        ar = torch.arange(S)\n        dense = torch.zeros(S, S, dtype=torch.float64)\n        dense.index_put_((ar[:, None].expand(-1, 8), idx[0]), prob[0], accumulate=True)\n        t1 = bench(lambda: ns[\"eval_direct\"](dense, R[:, 0], 0.95), repeat=1, number=1)\n        t2 = bench(lambda: ns[\"eval_iterative\"](idx[0], prob[0], R[:, 0], 0.95, 1e-8), repeat=1, number=1)\n        rows.append([S, fmt_time(t1), fmt_time(t2)])\n    R_.table(\"Politika değerlendirme (k = 8 seyrek geçiş)\", [\"S\", \"doğrudan çözüm O(S³)\", \"seyrek yineleme\"], rows,\n             note=\"Büyük S'de doğrudan çözüm kübik büyür; seyrek yineleme tarama başına O(S·k). MPI bu yüzden pratikte tercih edilir.\")\n",
 "x1": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rows = []\n        for S in (5, 20, 80, 200):\n            p = need(ns, \"pi_iterations\")(S, 4, 0.95, 5, 1)\n            v = need(ns, \"vi_iterations\")(S, 4, 0.95, 5, 1)\n            rows.append([S, f\"{np.mean(p):.1f} (en çok {max(p)})\", f\"{np.mean(v):.0f}\"])\n        ensure(all(float(r[1].split()[0]) < 15 for r in rows), \"PI birkaç iyileştirmede bitmeli\")\n        R_.table(\"Rastgele MDP, 4 eylem, γ = 0.95\", [\"S\", \"PI iyileştirme\", \"VI iterasyon (ε = 1e-8)\"], rows,\n                 note=\"Politika sayısı 4^S ile patlasa da PI pratikte durum sayısıyla çok yavaş büyüyen sayıda adım atar.\")\n    R_.test(\"ölçeklenme\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    rng = np.random.default_rng(0)\n\n    def t():\n        worst = 0\n        for _ in range(200):\n            S, A = 12, 3\n            P = rng.random((A, S, S)) ** 3; P /= P.sum(-1, keepdims=True)\n            R = rng.normal(size=(S, A)); g = rng.uniform(0.5, 0.99)\n            pi = rng.integers(0, A, S)\n            dV, dQ = need(ns, \"check_improvement\")(P, R, g, pi)\n            ensure(dV.min() >= -1e-9 and dQ.min() >= -1e-9, \"iyileştirme değeri düşürmemeli\")\n            new = need(ns, \"random_improving_policy\")(P, R, g, pi, int(rng.integers(1e6)))\n            idx = np.arange(S)\n            V0 = np.linalg.solve(np.eye(S) - g * P[pi, idx], R[idx, pi]); V1 = np.linalg.solve(np.eye(S) - g * P[new, idx], R[idx, new])\n            worst = min(worst, (V1 - V0).min())\n        ensure(worst >= -1e-9, \"tek durumlu iyileştirme de değeri düşürmemeli\")\n        return \"200 rastgele MDP'de hem tam açgözlü hem tek durumlu değişiklik hiçbir durumun değerini düşürmedi\"\n    R_.test(\"politika iyileştirme teoremi\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        V, pi = need(ns, \"gambler_vi\")(0.4)\n        ensure(V[0] == 0 and V[100] == 1 and np.all(np.diff(V) >= -1e-12), \"V monoton artan\")\n        close(V[50], 0.4, atol=1e-9, what=\"V(50) = p_h (hepsini yatır)\")\n        close(V[25], 0.4 * 0.4, atol=1e-9, what=\"V(25) = p_h²\")\n        ensure(pi[50] == 50 and pi[25] == 25 and pi[75] == 25, f\"π(50) = {pi[50]}, π(25) = {pi[25]}, π(75) = {pi[75]}\")\n        V2, pi2 = ns[\"gambler_vi\"](0.55)\n        ensure(np.all(pi2[1:100] == 1), \"p_h > 0.5'te en iyisi hep 1 yatırmak\")\n        R_.table(\"Kumarbaz problemi, p_h = 0.4\", [\"sermaye\", \"V\", \"π (bahis)\"], [[s, f\"{V[s]:.4f}\", pi[s]] for s in (1, 10, 12, 25, 37, 50, 51, 75, 99)],\n                 note=\"p_h < 0.5'te oyun aleyhte: az sayıda büyük bahisle 'cesur oyun' en iyisidir; politika 12.5, 25, 50 gibi noktalarda dikenlidir.\")\n    R_.test(\"kumarbaz problemi\", t)\n",
 "x4": "import numpy as np\nfrom labcheck import close,need,ensure\n\ndef check(ns,R_):\n    def tails():\n        T,r=need(ns,\"location\")(3,3)\n        close(np.asarray(T).sum(1),np.ones(21),atol=1e-12,what=\"Poisson kuyruklarıyla olasılık toplamı\")\n        close(r[0],0,atol=1e-12,what=\"araba yoksa kira yok\")\n    R_.test(\"Kuyruk kütlesi korunur\",tails)\n    def models():\n        base=need(ns,\"jack_modified\")(modified=False);new=ns[\"jack_modified\"]()\n        close(base[\"V\"][0,0],421.4140633965,atol=1e-6,what=\"özgün 441 durumlu çözüm\")\n        close(new[\"V\"][0,0],429.9463049642,atol=1e-6,what=\"ücretsiz taşıma + otopark\")\n        close(new[\"V\"][20,20],603.5367009161,atol=1e-6,what=\"iki otoparkın maliyeti\")\n        ensure(np.asarray(new[\"policy\"]).shape==(21,21),\"tam politika boyutu\")\n        ensure(new[\"residual\"]<1e-7,\"Bellman artığı\")\n        ensure(len(base[\"history\"])<=6 and len(new[\"history\"])<=6,\"politika yakınsaması\")\n        R_.table(\"Kaynak ve Alıştırma4.7\",[\"model\",\"V(0,0)\",\"V(20,20)\"],[[\"özgün\",base[\"V\"][0,0],base[\"V\"][20,20]],[\"değişmiş\",new[\"V\"][0,0],new[\"V\"][20,20]]])\n    R_.test(\"İki tam model\",models)\n"
}

In [ ]:
import importlib, labcheck, checks_m11_3_policy_iteration as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Fikir:** değer yinelemesi önce değeri yakınsatır, politikayı en sonda çıkarır. Politikayı doğrudan iyileştirebiliriz:

1. **Değerlendir:** $V^{\pi_n}=(I-\gamma T_{\pi_n})^{-1}R_{\pi_n}$ (ya da yinelemeyle).
2. **İyileştir:** $\pi_{n+1}(s)=\arg\max_a\big[R(s,a)+\gamma\sum_{s'}P(s'|s,a)V^{\pi_n}(s')\big]$.
3. Politika değişmeyene dek tekrarla.

**Lemma (politika iyileştirme):** $V^{\pi_{n+1}}\ge V^{\pi_n}$ (eleman bazında). İspat fikri: $H_{\pi_{n+1}}V^{\pi_n}=H^*V^{\pi_n}\ge H_{\pi_n}V^{\pi_n}=V^{\pi_n}$; $H_{\pi_{n+1}}$ monoton olduğu için tekrar uygulandıkça artar ve sabit noktası $V^{\pi_{n+1}}$'e yakınsar. **Teorem:** PI sonlu adımda $\pi^*$, $V^*$'a ulaşır (politika sayısı sonlu, döngü yok).

**Karşılaştırma:** VI ucuz iterasyonlar, çok iterasyon; PI pahalı iterasyonlar ($O(S^3)$ çözüm), çok az iterasyon (genelde < 10).

**Değiştirilmiş PI (MPI):** değerlendirmeyi kısalt: tam çözüm yerine $V\leftarrow H_{\pi}^{(k)}V$ ($k$ tarama), sonra iyileştir. $k=1$ → VI, $k=\infty$ → PI. Uygun $k$ ikisinden de hızlı olabilir. Durma: $\lVert V_n-V_{n-1}\rVert_\infty\le\varepsilon$ ise aynı türden sınır geçerlidir.

## 1 · Elle

<!-- çözüm -->

$\pi_0$=(kal, kal): $V(s_0)=0+0.5V(s_0)=0$, $V(s_1)=2+0.5V(s_1)=4$. İyileştirme: $s_0$'da git $1+0.5\cdot4=3>0$; $s_1$'de kal $4>0$. $\pi_1$=(git, kal): $V(s_0)=1+0.5\cdot4=3$. İkinci iyileştirme aynı politikayı verir → dur.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**İki durum, $\gamma=0.5$: $s_0$ kal → 0, $s_0$; git → 1, $s_1$. $s_1$ kal → 2, $s_1$; git → 0, $s_0$. Başlangıç $\pi_0$ = (kal, kal)**

- `v00`: $V^{\pi_0}(s_0)$
- `v01`: $V^{\pi_0}(s_1)$
- `q0`: İyileştirmede $Q(s_0,\text{git})$
- `v10`: $\pi_1$ = (git, kal) için $V(s_0)$
- `it`: Politika değişmeyene dek kaç iyileştirme adımı (değişmediğini doğrulayan dahil)?

**4 × 4 ızgara dünyası (Sutton Örnek 4.1), rastgele politika, $\gamma=1$, adım −1**

- `g1`: Köşe terminale komşu durum 1'in değeri

In [ ]:
cevaplar = dict(
    v00=None,
    v01=None,
    q0=None,
    v10=None,
    it=None,
    g1=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"v00":[0,0],"v01":[4,0],"q0":[3,0],"v10":[3,0],"it":[2,0],"g1":[-14,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, Sutton'ın 4 × 4 ızgarası: durumlar 0..15 (satır sırası), 0 ve 15 terminal (değer 0), eylemler yukarı/aşağı/sol/sağ (0..3), duvara çarpınca yerinde kal, her adım −1.

- `grid_step(s, a)` → `(s2, r)`.
- `evaluate(pi, gamma=1.0, theta=1e-10)` — `pi[s]` eylem olasılıkları listesi (4 eleman); **yerinde** (in-place, Gauss–Seidel) taramalarla, en büyük değişim `< theta` olunca dur. `(V, sweeps)`.
- `improve(V, gamma=1.0)` — açgözlü deterministik politika (olasılık listesi biçiminde, eşitlikte küçük eylem).
- `policy_iteration(gamma=1.0)` — rastgele politikadan başla; `(V, pi, n_improvements)`.

In [ ]:
%%writefile stage_b.py
def grid_step(s, a):
    ...


def evaluate(pi, gamma=1.0, theta=1e-10):
    ...


def improve(V, gamma=1.0):
    ...


def policy_iteration(gamma=1.0):
    ...


V, sweeps = evaluate([[0.25] * 4 for _ in range(16)])
print([round(v) for v in V])

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, `P (A, S, S)`, `R (S, A)`. Her algoritma `(V, pi, iters, backups)` döndürür; **backup** = bir durumun bir kez Bellman ile güncellenmesi (bir tarama = S backup; doğrudan çözüm ≈ S² backup sayılsın: muhasebe kuralı).

- `vi(P, R, gamma, eps)` — durma $\lVert V_n-V_{n-1}\rVert_\infty\le\varepsilon$.
- `pi_exact(P, R, gamma)` — doğrudan çözümlü PI; başlangıç politikası hep 0.
- `mpi(P, R, gamma, k, eps)` — iyileştir, $k$ değerlendirme taraması; durma VI'daki gibi (iyileştirme sonrası yeni değer ile önceki değer farkı).

Kontrol, hepsinin aynı $V^*$'a ulaştığını ve $k$'ya göre muhasebeyi tablolar.

In [ ]:
%%writefile stage_c.py
import numpy as np


def vi(P, R, gamma, eps=1e-8):
    ...


def pi_exact(P, R, gamma):
    ...


def mpi(P, R, gamma, k, eps=1e-8):
    ...


rng = np.random.default_rng(0)
P = rng.random((3, 20, 20)); P /= P.sum(-1, keepdims=True)
print(pi_exact(P, rng.normal(size=(20, 3)), 0.95)[2])

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

**Jack'in araba kiralama problemi** (Sutton Örnek 4.2): iki şube, en çok 20'şer araba; kiralama talebi Poisson(3) ve Poisson(4), iadeler Poisson(3) ve Poisson(2); kiralama başı 10 \$; gece en fazla 5 araba taşınabilir, araba başı 2 \$; γ = 0.9. Sıra: taşı (21'e kırp) → talepler (en fazla eldeki kadar kiralanır) → iadeler (20'de kırp).

- `location(lam_req, lam_ret, N=20)` → `(T (N+1, N+1), r (N+1,))`: taşımadan sonra $n$ araba varken ertesi sabahki sayının dağılımı ve beklenen kira geliri. Kuyruk olasılıkları sona eklenir (talep ≥ n, iade ≥ N).
- `jack_pi(gamma=0.9)` → `(V (21, 21), pi (21, 21), iters)`: eylem $a\in[-5,5]$ (pozitif: 1. şubeden 2.'ye), uygulanabilirlik $a\le n_1$, $-a\le n_2$. Değerlendirme doğrudan çözüm (441 × 441; ortak geçiş ayrılabilir: `T1[m1][:, :, :, None] * T2[m2][:, :, None, :]`), iyileştirme tüm eylemler için `einsum`. Politika değişmeyince dur.

In [ ]:
%%writefile stage_d.py
import numpy as np
import torch
from scipy.stats import poisson


def location(lam_req, lam_ret, N=20):
    ...


def jack_pi(gamma=0.9):
    ...


V, pi, it = jack_pi()
print(it, pi[20, 0], round(float(V[0, 0]), 1))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Büyük MDP'de politika değerlendirmesi.

- `random_sparse_mdp(S, A, k, seed=0)` — her (s, a) için rastgele k hedef durumlu seyrek geçiş; `torch.sparse_coo` ya da (A, S, k) indeks + olasılık dizileri.
- `eval_direct(Pi_dense, r, gamma)` — `torch.linalg.solve`.
- `eval_iterative(idx, prob, r, gamma, tol)` — $V\leftarrow r+\gamma\sum_kp_k V[\text{idx}_k]$ (gather), yakınsayana dek; tarama sayısını da döndür.
- `mpi_sparse(idx, prob, R, gamma, k, eps)` — seyrek MPI.

Kontrol, S büyüdükçe doğrudan çözümün ($O(S^3)$) ve seyrek yinelemenin ($O(Sk)$ / tarama) süresini karşılaştırır.

In [ ]:
%%writefile stage_e.py
import torch


def random_sparse_mdp(S, A, k, seed=0):
    ...


def eval_direct(Pi_dense, r, gamma):
    ...


def eval_iterative(idx, prob, r, gamma, tol=1e-10):
    ...


def mpi_sparse(idx, prob, R, gamma, k, eps=1e-8):
    ...


idx, prob, R = random_sparse_mdp(100, 3, 5)
print(idx.shape, prob.shape, R.shape)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · PI'nin adım sayısı durum sayısıyla nasıl büyür?

- `pi_iterations(S, A, gamma, trials, seed)` — rastgele (yoğun) MDP'lerde doğrudan çözümlü PI'nin iyileştirme sayıları listesi.
- `vi_iterations(S, A, gamma, trials, seed, eps=1e-8)` — aynı MDP'lerde VI iterasyon sayıları.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def pi_iterations(S, A, gamma, trials, seed=0):
    ...


def vi_iterations(S, A, gamma, trials, seed=0, eps=1e-8):
    ...


print(pi_iterations(10, 2, 0.9, 3))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Politika iyileştirme teoremini sayısal doğrula

- `q_from_v(P, R, gamma, V)` — `(S, A)`.
- `check_improvement(P, R, gamma, pi, seed=0)` — $\pi'$ = $Q^\pi$'ye göre açgözlü; $V^{\pi'}-V^\pi$ (hep ≥ 0) ve $Q^\pi(s,\pi'(s))-V^\pi(s)$ (≥ 0) dizilerini döndür.
- `random_improving_policy(P, R, gamma, pi, seed=0)` — **yalnız bir durumda** $Q^\pi(s,a)>V^\pi(s)$ olan bir eyleme geçen politika; teorem bunun da iyileştirme olduğunu söyler.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def q_from_v(P, R, gamma, V):
    ...


def check_improvement(P, R, gamma, pi, seed=0):
    ...


def random_improving_policy(P, R, gamma, pi, seed=0):
    ...


print("iyileştirme")

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Kumarbaz problemi (Sutton Örnek 4.3)

**Kumarbaz problemi:** sermaye $s\in\{1..99\}$; her turda $a\in\{1..\min(s,100-s)\}$ yatır; $p_h=0.4$ olasılıkla $s+a$, aksi halde $s-a$; 100'e ulaşınca ödül 1 (terminal), 0'da biter. γ = 1.

- `gambler_vi(ph=0.4, eps=1e-12)` → `(V (101,), pi (101,))`: $V(0)=0$, $V(100)=1$; politika: eşitliklerde **en küçük** bahis değil, `np.round(Q, 9)` sonrası **en küçük** $a$ (kitaptaki "dikenli" politika sayısal eşitlik seçimine duyarlıdır).


Sutton Alıştırma4.9 (s.84–85): `ph=0.25` ve `ph=0.55` için ayrı çalıştır;değer ve politika dizilerini sermayeye karşı çiz. `eps=1e-6,1e-9,1e-12` ile en büyük V farkını ve değişen politika hücresi sayısını raporla. l5 bu grafikleri yerinde taramayla üretir;bu aşama eşzamanlıdır. Terminal V100=1 sınırını kullanırken ayrıca hedef ödülü ekleme. Kaynağın 0 bahis nüansı anlatımda açıklanır;bu aşama sonlanan pozitif bahis politikaları içindir.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def gambler_vi(ph=0.4, eps=1e-12):
    ...


V, pi = gambler_vi()
print(pi[1:100:7])

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

### E4 · Sutton 4.7: ücretsiz araba ve sabit otopark

**Sutton Alıştırma 4.7 (s.82):** d aşamasındaki özgün araba kiralama problemini önce yeniden üret, sonra iki değişikliği birlikte uygula. İlk şubeden ikinciye her gece ilk araba ücretsiz; ek arabalar ve ters yöndeki bütün arabalar 2 birim. Taşımadan sonra bir şubede 10 adetten fazla araba varsa o şube için sabit 4 otopark maliyeti var.

`location(req,ret,N=20)` → `(T,r)` ve `jack_modified(gamma=0.9,modified=True)` → `{V,policy,history,residual}`. V/policy biçimi (21,21); history her değerlendirilen politikanın kopyasını, değerini ve tarama sayısını taşır. `modified=False` önce Örnek4.2 modelini verir. Her kuyruk olasılığını son kovaya kat: talep karşılanabilen araç sayısında, iadeler kapasitede doyar. Kuyruğu atıp satır toplamını bozma.

Eşit en iyi eylemler arasında mevcut eylemi koru. Değerlendirmeyi önceki V ile başlat. Tam referans için 10⁻¹¹ tarama eşiği kullan; default γ=0.9. Kaynak Şekil4.2 ve değişmiş problem için l4 kaynak/4.7 ön ayarları politika sırasını ve değer yüzeyini üretir. Sayısal rapor ve yeniden üretim betiği `notes/sutton-2018` içindedir. Kod çözümü bütün 441 durumu kullanır; küçük bir kapasite örneği bu görevin yerine geçmez.

In [ ]:
%%writefile extra_x4.py
import numpy as np
import math


def location(req, ret, N=20):
    ...


def jack_modified(gamma=0.9, modified=True):
    ...


result = jack_modified()
print(result["policy"][20, 0], round(float(result["V"][0, 0]), 3))

In [ ]:
# Kontrol (E4)
labcheck.run_file("extra_x4.py", CHECKS["x4"])

## Makale ödevi: Politika yinelemesinin karmaşıklığı

[Ye — The Simplex and Policy-Iteration Methods Are Strongly Polynomial for the Markov Decision Problem with a Fixed Discount Rate (Math. of OR 2011)](https://doi.org/10.1287/moor.1110.0516)

Sabit γ için politika yinelemesinin (Howard) en fazla $O\\big(\\frac{|S||A|}{1-\\gamma}\\log\\frac{|S|}{1-\\gamma}\\big)$ iterasyonda bittiğini gösteren sonucu incele; MDP'yi doğrusal programa çevir, PI'yi simpleks yöntemiyle ilişkilendir ve deneyle kontrol et.


**Adımlar**

1. MDP'nin birincil LP'si: $\min\sum_s V(s)$ s.t. $V(s)\ge R(s,a)+\gamma\sum P V$ — `scipy.optimize.linprog` ile çöz, VI ile karşılaştır.
2. İkili LP: ziyaret ölçüleri $x(s,a)$; PI'nin bir iyileştirme adımının simpleks pivotlarına karşılığı.
3. Rastgele ve kötü durum (Melekopoglou–Condon tipi) MDP'lerde |S|, γ'ya karşı iterasyon grafikleri.

**Kabul ölçütleri**

- LP, VI ve PI aynı V*'ı verir (1e-8).
- γ → 1 iken iterasyon sayısının 1/(1−γ) ile büyüdüğü bir aile gösterilmiş.

## Kaynaklar

- [Sutton & Barto — Reinforcement Learning: An Introduction (2. baskı), Bölüm 4](http://incompleteideas.net/book/the-book-2nd.html) — CC BY-NC-ND 2.0
- [Puterman & Shin — Modified Policy Iteration Algorithms for Discounted MDPs (Management Science 1978)](https://doi.org/10.1287/mnsc.24.11.1127) — ©
- [Howard — Dynamic Programming and Markov Processes (MIT Press 1960)](https://mitpress.mit.edu/) — ©
- [Poupart — CS885 Reinforcement Learning, Ders 3](https://cs.uwaterloo.ca/~ppoupart/teaching/cs885-spring20/) — doğrulanmadı